In [0]:
-- Question 1: How Many Customers has Foodie-Fi ever had?

SELECT count(distinct(customer_id)) as total_customers FROM subscriptions;

--- Question 2: What is the monthly distribution of trial plan start_date values for our dataset - use the start of the month as the group by value

SELECT 
to_date(date_trunc('month', s.start_date)) as month,
count(case when p.plan_name = 'trial' then 1 end) as trial_customers
FROM subscriptions s  
JOIN plans p on s.plan_id = p.plan_id
GROUP BY month
ORDER BY month;

-- Question 3: What plan start_date values occur after the year 2020 for our dataset? Show the breakdown by count of events for each plan_name

SELECT
p.plan_name,
count(*) as total_customers
FROM subscriptions s  
JOIN plans p on s.plan_id = p.plan_id
WHERE to_date(date_trunc('month', s.start_date)) > '2020-12-31' 
GROUP BY p.plan_name;


-- Question 4: What is the customer count and percentage of customers who have churned rounded to 1 decimal place?

SELECT count(case when p.plan_name = 'churn' then 1 end) as churned_customers,
round((count(case when p.plan_name = 'churn' then 1 end)/count(distinct(s.customer_id)))*100, 2) as percentage_churned,
count(distinct s.customer_id) as no_of_customers
FROM plans p 
JOIN subscriptions s on p.plan_id = s.plan_id;

-- Question 5: How many customers have churned straight after their initial free trial - what percentage is this rounded to the nearest whole number?

WITH cte as (SELECT s.customer_id,
p.plan_name,
s.start_date,
LAG(p.plan_name, 1) OVER (PARTITION BY s.customer_id ORDER BY s.start_date) as previous_plan
FROM subscriptions s 
JOIN plans p ON p.plan_id = s.plan_id),

churned_customers as (SELECT distinct customer_id from cte
WHERE previous_plan = 'trial' and plan_name = 'churn'),

total_customers as (SELECT distinct customer_id from subscriptions)

SELECT
      (SELECT count(*) FROM churned_customers) as churned,
      (SELECT count(*) FROM total_customers) as total,
      (round((SELECT count(*) FROM churned_customers) / (SELECT count(*) FROM total_customers) *100, 2)) as percentage_churned_after_trial

-- Question 6: What is the number and percentage of customer plans after their initial free trial?

WITH cte as (SELECT *,
LEAD(plan_id, 1) OVER (PARTITION BY customer_id ORDER BY start_date) as next_plan
FROM subscriptions),

customers as (SELECT distinct customer_id FROM subscriptions),

total_customers as (SELECT next_plan as plan_after_trial,
count(*) total_plans 
FROM cte
WHERE plan_id = 0
GROUP BY next_plan)

SELECT plan_after_trial,
total_plans,
round(total_plans/(SELECT count(*) FROM customers)*100, 2) as percentage
FROM total_customers
ORDER BY percentage DESC;


-- Question 7: What is the customer count and percentage breakdown of all 5 plan_name values at 2020-12-31?

WITH cte as (SELECT customer_id, max(plan_id) as plan_id, max(start_date) as start_date FROM subscriptions
WHERE start_date <= '2020-12-31'
GROUP BY customer_id),

customer_count as (
SELECT plan_id, 
count(*) as total_customers
FROM cte
GROUP BY plan_id),

customers as (SELECT distinct customer_id FROM cte
WHERE start_date <= '2020-12-31')

SELECT plan_id,
total_customers,
round(total_customers/(SELECT count(*) FROM customers) * 100, 2) as percentage
FROM customer_count
ORDER BY percentage DESC;


-- Question 8: How many customers have upgraded to an annual plan in 2020?

SELECT p.plan_name, count(distinct a.customer_id) as upgraded_customers FROM subscriptions a  
JOIN plans p on p.plan_id = a.plan_id
JOIN subscriptions b on b.customer_id = a.customer_id
WHERE a.start_date BETWEEN '2020-01-01' AND '2020-12-31'
AND b.plan_id < a.plan_id AND
a.plan_id = 3
AND b.start_date < a.start_date
GROUP BY p.plan_name;


-- Question 9: How many days on average does it take for a customer to an annual plan from the day they join Foodie-Fi?

SELECT
ROUND(AVG(DATEDIFF(day, b.start_date, a.start_date)), 0) as difference
FROM subscriptions a  
JOIN subscriptions b ON b.customer_id = a.customer_id
WHERE a.plan_id = 3 
AND b.plan_id = 0
AND b.start_date < a.start_date

-- Question 10: Can you further breakdown this average value into 30 day periods (i.e. 0-30 days, 31-60 days etc)

WITH differences as (SELECT
DATEDIFF(day, b.start_date, a.start_date) as difference
FROM subscriptions a  
JOIN subscriptions b ON b.customer_id = a.customer_id
WHERE a.plan_id = 3 
AND b.plan_id = 0
AND b.start_date < a.start_date)

SELECT
  CASE 
    WHEN difference BETWEEN 0 AND 30 THEN '0 to 30 days'
    WHEN difference BETWEEN 31 AND 60 THEN '31 to 60 days'
    WHEN difference BETWEEN 61 AND 90 THEN '61 to 90 days'
    WHEN difference BETWEEN 91 AND 120 THEN '91 to 120 days'
    WHEN difference BETWEEN 121 AND 150 THEN '121 to 150 days'
    WHEN difference BETWEEN 151 AND 180 THEN '151 to 180 days'
    ELSE '181 days plus'
  END AS period,
  COUNT(*) AS customers_in_period,
  ROUND(AVG(difference), 0) AS average_days_in_period
FROM differences
GROUP BY period
ORDER BY MIN(difference);

-- Question 11: How many customers downgraded from a pro monthly to a basic monthly plan in 2020?

SELECT count(*) as num_downgrade FROM subscriptions a
JOIN subscriptions b ON b.customer_id = a.customer_id
WHERE a.plan_id = 1 
AND b.plan_id = 2
AND a.plan_id > b.plan_id
AND a.start_date BETWEEN '2020-01-01' AND '2020-12-31';








